# Toy Data Poisoning Demo (2D)

This notebook demonstrates **training-time poisoning** on a simple 2D binary classification task.

We show two variants:
- **Availability poisoning**: degrade global accuracy by corrupting labels.
- **Targeted poisoning**: cause a specific target point to be misclassified by adding a few crafted points.

## Run it
Run all cells top-to-bottom. No downloads required.


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
def make_blobs(n_per_class=200, std=0.9, centers=((-2.0, 0.0), (2.0, 0.0))):
    c0 = np.array(centers[0], dtype=np.float32)
    c1 = np.array(centers[1], dtype=np.float32)
    x0 = c0 + std * np.random.randn(n_per_class, 2).astype(np.float32)
    x1 = c1 + std * np.random.randn(n_per_class, 2).astype(np.float32)
    X = np.concatenate([x0, x1], axis=0)  # (N, 2)
    y = np.concatenate([
        np.zeros((n_per_class, 1), dtype=np.float32),
        np.ones((n_per_class, 1), dtype=np.float32),
    ], axis=0)  # (N, 1)
    return X, y


X_np, y_np = make_blobs(n_per_class=250, std=1.0)
perm = np.random.permutation(len(X_np))
X_np, y_np = X_np[perm], y_np[perm]

# Tensors: X: (N, 2), y: (N, 1)
X = torch.tensor(X_np, device=device)
y = torch.tensor(y_np, device=device)

X.shape, y.shape

In [ ]:
def train_logreg(X_train, y_train, *, epochs=800, lr=0.15, weight_decay=0.0):
    """Logistic regression in PyTorch.
    X_train: (N, 2)
    y_train: (N, 1)
    """
    model = nn.Linear(2, 1).to(X_train.device)
    opt = torch.optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.BCEWithLogitsLoss()

    for _ in range(epochs):
        logits = model(X_train)  # (N, 1)
        loss = loss_fn(logits, y_train)  # scalar
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
    return model


@torch.no_grad()
def accuracy(model, X_eval, y_eval):
    """X_eval: (N, 2), y_eval: (N, 1)"""
    probs = torch.sigmoid(model(X_eval))  # (N, 1)
    preds = (probs >= 0.5).float()  # (N, 1)
    return float((preds == y_eval).float().mean().item())


def plot_boundary(model, X_plot, y_plot, title, *, extra_points=None):
    """Decision boundary plot.
    X_plot: (N, 2)
    y_plot: (N, 1)
    extra_points: dict with keys {"X": (K,2), "c": color, "label": str}
    """
    X_cpu = X_plot.detach().cpu().numpy()
    y_cpu = y_plot.detach().cpu().numpy().reshape(-1)

    x_min, x_max = X_cpu[:, 0].min() - 1.0, X_cpu[:, 0].max() + 1.0
    y_min, y_max = X_cpu[:, 1].min() - 1.0, X_cpu[:, 1].max() + 1.0

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 250, dtype=np.float32),
        np.linspace(y_min, y_max, 250, dtype=np.float32),
    )
    grid = np.stack([xx.ravel(), yy.ravel()], axis=1).astype(np.float32)  # (G, 2)
    grid_t = torch.tensor(grid, device=X_plot.device)  # (G, 2)

    with torch.no_grad():
        probs = torch.sigmoid(model(grid_t)).view(xx.shape)  # (H, W)
    Z = probs.detach().cpu().numpy()

    plt.figure(figsize=(7, 5))
    plt.contourf(xx, yy, Z, levels=30, cmap="coolwarm", alpha=0.35)
    plt.scatter(X_cpu[:, 0], X_cpu[:, 1], c=y_cpu, cmap="coolwarm", s=14, alpha=0.9)
    plt.contour(xx, yy, Z, levels=[0.5], colors=["white"], linewidths=2)

    if extra_points is not None:
        X_ex = extra_points["X"].detach().cpu().numpy()  # (K, 2)
        plt.scatter(X_ex[:, 0], X_ex[:, 1], c=extra_points.get("c", "yellow"), s=55, marker="x", linewidths=2, label=extra_points.get("label", "extra"))
        plt.legend(loc="upper right")

    plt.title(title)
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.tight_layout()
    plt.show()


In [ ]:
baseline = train_logreg(X, y, epochs=800, lr=0.15)
acc_base = accuracy(baseline, X, y)
print(f"Baseline training accuracy: {acc_base:.3f}")
plot_boundary(baseline, X, y, "Baseline (clean training)")

In [ ]:
# Availability poisoning: flip labels of a small subset.
# y: (N, 1)
N = X.shape[0]
k_flip = 30
idx = torch.randperm(N, device=device)[:k_flip]

y_avail = y.clone()  # (N, 1)
y_avail[idx] = 1.0 - y_avail[idx]

avail_model = train_logreg(X, y_avail, epochs=800, lr=0.15)
acc_avail_on_clean = accuracy(avail_model, X, y)
print(f"After availability poisoning: accuracy on clean labels = {acc_avail_on_clean:.3f}")

plot_boundary(
    avail_model,
    X,
    y,
    f"Availability poisoning (flip {k_flip} labels)",
    extra_points={"X": X[idx], "c": "yellow", "label": "flipped labels"},
)

In [ ]:
# Targeted poisoning: add a small cluster near a chosen target point to push it across the boundary.
# x_target: (1, 2)
x_target = torch.tensor([[-0.4, 0.0]], device=device, dtype=X.dtype)

with torch.no_grad():
    p_before = float(torch.sigmoid(baseline(x_target)).item())  # scalar
print(f"Target point predicted P(class=1) before poisoning: {p_before:.3f}")

k_poison = 15
sigma = 0.25
X_poison = x_target + sigma * torch.randn((k_poison, 2), device=device, dtype=X.dtype)  # (K, 2)
y_poison = torch.ones((k_poison, 1), device=device, dtype=y.dtype)  # (K, 1)

# Augmented training set
X_aug = torch.cat([X, X_poison], dim=0)  # (N+K, 2)
y_aug = torch.cat([y, y_poison], dim=0)  # (N+K, 1)

targeted_model = train_logreg(X_aug, y_aug, epochs=900, lr=0.14)

with torch.no_grad():
    p_after = float(torch.sigmoid(targeted_model(x_target)).item())  # scalar
print(f"Target point predicted P(class=1) after poisoning:  {p_after:.3f}")

plot_boundary(
    targeted_model,
    X,
    y,
    f"Targeted poisoning (add {k_poison} points near a target)",
    extra_points={"X": torch.cat([x_target, X_poison], dim=0), "c": "yellow", "label": "target + poison"},
)